Testando frases com modelo BERTimbau pré-treinado.



1. Carregar modelo do Angelo

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

repo_id = "angelosbc/bertimbau-steam-sentiment"

# Baixa e carrega o tokenizador e os pesos ajustados do Hugging Face
tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModelForSequenceClassification.from_pretrained(repo_id)

# Envia o modelo para a GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Modelo carregado com sucesso no dispositivo: {device}")

config.json:   0%|          | 0.00/954 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/678k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo carregado com sucesso no dispositivo: cpu


Programa

In [6]:
import torch

# Mapeamento das classes do dataset
id2label = {0: "Não Recomenda", 1: "Recomenda"}

def analisar_review():
    print("=" * 60)
    print("Classificador Steam: Predicao de Sentimento / Ironia")
    print("Digite 'sair' para encerrar o programa.")
    print("=" * 60)

    model.eval()

    while True:
        texto = input("\nDigite a analise do jogo: ").strip()

        if texto.lower() in ["sair", "exit", "quit"]:
            print("\nEncerrando o programa.")
            break

        if not texto:
            print("Digite algum texto para testar.")
            continue

        # Tokenizacao
        inputs = tokenizer(
            texto,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        # Inferencia
        with torch.no_grad():
            outputs = model(**inputs)
            probabilidades = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

        # Probabilidades
        prob_nao_rec = probabilidades[0] * 100
        prob_rec = probabilidades[1] * 100
        classe_predita = 1 if prob_rec > prob_nao_rec else 0

        # Identificacao de conflito semantico / sarcasmo
        gatilhos_ironia = ["maravilhoso", "excelente", "otimo", "ótimo", "recomendo super", "bom demais"]
        gatilhos_problema = ["travou", "bug", "injogavel", "injogável", "fechou", "nao abre", "não abre", "crash", "reembolso"]

        texto_lower = texto.lower()
        tem_elogio = any(t in texto_lower for t in gatilhos_ironia)
        tem_critica = any(t in texto_lower for t in gatilhos_problema)

        print("\n" + "-" * 40)
        print(f"Resultado: {id2label[classe_predita]}")
        print(f" - Nao Recomenda: {prob_nao_rec:.2f}%")
        print(f" - Recomenda:     {prob_rec:.2f}%")

        if tem_elogio and tem_critica:
            print("\n[ALERTA] Conflito semantico / possivel sarcasmo detectado.")
            print("         O texto combina termos de elogio com indicacao de falha tecnica.")
        print("-" * 40)

# Executa o programa interativo
analisar_review()

Classificador Steam: Predicao de Sentimento / Ironia
Digite 'sair' para encerrar o programa.

Digite a analise do jogo:  Nota 10! Recomendo muito! Jogo perfeito para jogar no lixo! Parabéns ao envolvidos!

----------------------------------------
Resultado: Recomenda
 - Nao Recomenda: 6.59%
 - Recomenda:     93.41%
----------------------------------------

Digite a analise do jogo: sair

Encerrando o programa.
